# BERTopic grid search v2: OPF - Scopus (15,743 papers)

Grid of `GRID_V2_OUTLIER8_PLAN.pdf` (Table 3 v2), scored and ranked on 8 criteria.

| Parameter | Values | Count |
|---|---|---|
| Embedding model | paraphrase-mpnet-base-v2, BioBERT-STS, MiniLM | 3 |
| n_neighbors | 10-50, step 5 | 9 |
| n_components | 10-50, step 5 | 9 |
| min_cluster_size | 0.35, 0.70, 1.00, 1.35, 1.70, 2.00, 2.35, 2.70, 3.00 % of papers | 9 |
| min_samples | 10-60, step 10 | 6 |
| cluster_selection_method | eom, leaf | 2 |
| **Settings** | 3 x 9 x 9 x 9 x 6 x 2 | **26,244** |

Fixed: min_dist 0, UMAP cosine, HDBSCAN euclidean, seed 123, 10 label words from the cleaned stems, outliers kept.

**Ranking (0-3 points per criterion, total out of 24).** Eligible = 5-30 topics.
* Outlier: <= 3% -> 3, > 3-6% -> 2, > 6-8% -> 1, > 8% -> 0.
* Label match (kappa), Label coverage, NPMI, Cohesion, Diversity, Size (largest topic %, smaller is better), Stability (mean seed ARI):
  best quarter of the eligible settings -> 3 ... worst quarter -> 0.
* Ties: lower mean rank. Stability (seeds 0, 1, 2) is measured for the 60 best settings before stability; the top 20 comes from these.

**Steps:** 1 environment, 2 grid definition, 3 run the grid, 4 progress, 5 rank, 6 stability, 7 final ranking and top 20,
8 the three best settings (groups, labels, members), 9 hand checks.

**Checkpoints.** Every setting writes its own `summary.json` the moment it is finished, and every evaluation step writes a marker in
`progress/`; all steps skip finished work, so the notebook can be rerun from any cell after an interruption.
`grid_v2_watchdog.py` (run outside the notebook) checks the processes every 5 minutes, restarts a dead one and saves a continuity
checkpoint after each step.

## Step 1. Environment and paths

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path(r"D:/Journal SLR_Scopus_Overton")
os.chdir(ROOT)
HERE = ROOT / "Reports/2026-10-17_bertopic_tuning/grid_v2_outlier8"
sys.path[:0] = [str(HERE), str(HERE.parent)]
os.environ["PYTHONPATH"] = "D:/python_env/sciimpact_snowball22"     # snowballstemmer 2.2.0, as every saved output
PY = "D:/python_env/bertopic_tuning/Scripts/python.exe"
DB, REVIEW, PAPERS = "opf_scopus", "OPF", 15743
GRID_DIR = ROOT / "Codes/output/bertopic_tuning" / DB / "grid_v2"
import run_grid_v2 as G
import evaluate_grid_v2 as E
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)
print(sys.executable)

## Step 2. The grid
Every setting with its min_cluster_size in papers: max(10, round(share x papers)).

In [ ]:
import itertools
grid = pd.DataFrame([{"Grid_id": G.grid_id(e, nn, nc, m, ms, s), "embedder": e, "n_neighbors": nn, "n_components": nc,
                      "mcs_share_pct": m, "min_cluster_size": max(10, int(round(m / 100 * PAPERS))), "min_samples": ms, "selection": s}
                     for e, nn, nc, m, ms, s in itertools.product(G.EMBS, G.NN, G.NC, G.MCS, G.MS, G.SEL)])
assert len(grid) == 26244
print(len(grid), "settings")
grid.drop_duplicates("mcs_share_pct")[["mcs_share_pct", "min_cluster_size"]]

## Step 3. Run the grid (long)
`RUN_GRID = True` starts 9 processes (3 models x 3 n_neighbors groups) through the continuity command wrapper, as
`launch_grid_v2.ps1` does. Finished settings are skipped, so starting again only completes the missing ones.
Leave it `False` while the command-line run is still going.

In [ ]:
RUN_GRID = False
groups = {"a": [10, 15, 20], "b": [25, 30, 35], "c": [40, 45, 50]}
if RUN_GRID:
    env = dict(os.environ, NUMBA_NUM_THREADS="2", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
    procs = []
    for emb in G.EMBS:
        for g, nns in groups.items():
            cid = f"grid-v2-{DB}-{emb}-nn{g}-nb".replace("_", "-")
            args = [PY, ".claude/skills/workspace-continuity/scripts/checkpoint.py", "run", "--id", cid, "--", PY, "-X", "utf8",
                    str(HERE / "run_grid_v2.py"), "--db", DB, "--embedder", emb, "--nn", *map(str, nns)]
            procs.append(subprocess.Popen(args, cwd=ROOT, env=env))
    print(len(procs), "processes started; follow progress with Step 4")

## Step 4. Progress (rerun this cell to refresh)

In [ ]:
done = sum(1 for p in GRID_DIR.glob("*/summary.json")) if GRID_DIR.exists() else 0
print(f"{done:,} / 26,244 settings finished ({100 * done / 26244:.1f}%)")
status = HERE / "progress" / "status.json"
if status.exists():
    s = json.loads(status.read_text())
    r = s["reviews"][DB]
    print("watchdog check", s["checked_at"], "| rate", r["settings_per_hour"], "settings/hour | ETA", r["eta_hours"], "h | flags", s["flags"])
    display(pd.DataFrame(r["groups"]))

## Step 5. Rank without stability
Collects all summaries, applies the 5-30 topic rule and gives the points of 7 criteria.

In [ ]:
assert done == 26244, "grid not complete yet"
E.step_rank(DB)
pre = pd.read_csv(E.TABLES / f"{DB}_ranked_pre.csv")
el = pre[pre["Eligible"]]
print(len(pre), "settings,", len(el), "eligible (5-30 topics)")
print("Outlier points of the eligible settings:", el["Outlier_pts"].value_counts().sort_index().to_dict())
print("Lowest outlier share among eligible settings:", el["Outlier_pct"].min(), "%")
el.sort_values("Rank").head(10)[["Rank", "Grid_id", "Topics", "Outlier_pct", "Largest_topic_pct", "Label_match_kappa", "Total_points"]]

## Step 6. Stability of the 60 leading settings (seeds 0, 1, 2)
Six parallel parts; finished seeds are skipped.

In [ ]:
env = dict(os.environ, NUMBA_NUM_THREADS="3", OMP_NUM_THREADS="2")
procs = [subprocess.Popen([PY, "-X", "utf8", str(HERE / "evaluate_grid_v2.py"), "stability", "--db", DB, "--part", str(i), "--parts", "6"],
                          cwd=ROOT, env=env) for i in range(6)]
codes = [p.wait() for p in procs]
assert not any(codes), codes

## Step 7. Final ranking on 8 criteria and the top 20

In [ ]:
E.step_final(DB)
top = pd.read_csv(E.TABLES / f"{DB}_top20.csv")
pts = ["Label match_pts", "Label coverage_pts", "NPMI_pts", "Cohesion_pts", "Diversity_pts", "Size_pts", "Stability_pts", "Outlier_pts"]
top[["Rank", "Grid_id", "Topics", "Outlier_pct", "Label_match_kappa", "Label_coverage_pct", "NPMI", "Cohesion", "Diversity_pct",
     "Largest_topic_pct", "Stability_ARI", *pts, "Total_points", "Mean_rank"]]

## Step 8. The three best settings: groups, labels and members (as T2_SMALL_TOPIC_OPTIONS)

In [ ]:
E.step_top3(DB)
grp = pd.read_csv(E.TABLES / f"{DB}_top3_groups.csv")
for (rank, gid), g in grp.groupby(["Rank", "Grid_id"]):
    N = int(g["N_papers_with_topic"].iloc[0])
    print(f"Rank {rank}: {gid} - {len(g)} topics, {N:,} papers with a topic, outliers {PAPERS - N:,} ({100 * (PAPERS - N) / PAPERS:.1f}%)")
    display(g[["Topic", "Label_words", "Papers_in_topic_r", "Pct_of_papers_with_topic", "Pct_of_all_papers", "Papers_words_point_here_c",
               "Recall_pct", "Precision_pct", "Topic_kappa", "Closest_titles"]])

In [ ]:
mem = pd.read_csv(E.TABLES / f"{DB}_top3_members.csv")
mem[mem["Rank"] == 1].head(30)          # closest and random member papers of the best setting, for checking by hand

## Step 9. Hand checks

In [ ]:
r = top.iloc[0]
print("Outlier points:", r["Outlier_pct"], "% ->", E.outlier_points(r["Outlier_pct"]), "(table:", int(r["Outlier_pts"]), ")")
fin = pd.read_csv(E.TABLES / f"{DB}_ranked.csv")
el = fin[fin["Eligible"]]
q = el["Label_match_kappa"].quantile([0.25, 0.5, 0.75])
print("Kappa quartiles of the eligible settings:", q.round(4).to_dict(), "| best setting kappa", r["Label_match_kappa"], "->", int(r["Label match_pts"]), "points")
print("Total =", " + ".join(str(int(r[p])) for p in pts), "=", int(sum(r[p] for p in pts)), "(table:", int(r["Total_points"]), ")")
g1 = grp[grp["Rank"] == 1].iloc[0]
N, rr, c, a = int(g1.N_papers_with_topic), int(g1.Papers_in_topic_r), int(g1.Papers_words_point_here_c), int(g1.In_both_a)
po, pe = (N - rr - c + 2 * a) / N, (rr * c + (N - rr) * (N - c)) / N ** 2
print(f"Topic {int(g1.Topic)} kappa by hand: ({po:.4f} - {pe:.4f}) / (1 - {pe:.4f}) = {(po - pe) / (1 - pe):.3f} (table: {g1.Topic_kappa})")